# Problem 5.X1 — Huffman Coding of 8-Bit Sampled Music

*ECE 646, Homework 5* — a from-scratch (binary) Huffman implementation that losslessly compresses `music3b.wav`.

**How to run:** *Runtime → Run all*. When the file-chooser widget appears in **Step 1**, select `music3b.wav` (it ships
in the GitHub repo / zip that came with this notebook). Everything else is automatic —
`numpy`, `scipy` and `matplotlib` are preinstalled in Colab.

The implementation files written in **Step 4** below are byte-for-byte the ones in the repository
(`huffman.py`, `utils.py`); this notebook just walks the exact same pipeline so you can watch it work.


## Step 1 — Read the music file

The file is 8-bit PCM (`uint8` sample values, i.e. 0–255), which is what makes it directly Huffman-codable.


In [ ]:
from google.colab import files
import os

# Already in the session? (e.g. you dragged it into the file browser) Then skip the widget.
if os.path.exists("music3b.wav"):
    print("music3b.wav already present — skipping upload.")
else:
    uploaded = files.upload()   # a chooser widget appears — select music3b.wav
    if uploaded:
        with open("music3b.wav", "wb") as f:
            f.write(next(iter(uploaded.values())))
    assert os.path.exists("music3b.wav"), \
        "music3b.wav not found. Re-run this cell and select it in the chooser."
print("OK")


## Step 2 — Calculate the empirical distribution

Count how many times each sample value occurs; frequencies *over total* = empirical probabilities.


In [ ]:
import numpy as np
from scipy.io import wavfile

rate, data = wavfile.read("music3b.wav")
print(f"Sampling rate: {rate} Hz | samples: {len(data)} | dtype: {data.dtype}")

symbols, counts = np.unique(data, return_counts=True)
probabilities = counts / counts.sum()
empirical = {sym: p for sym, p in zip(symbols, probabilities)}
print(f"{len(symbols)} distinct sample values")
print("most common:", [(int(s), f"{p:.3f}") for s, p in empirical.items()][:5])


## Step 3 — Entropy of the file (iid assumption)

Shannon entropy of the empirical distribution, treating samples as iid.


In [ ]:
import numpy as np

# The definition of Shannon entropy, written out here (it is utils.entropy —
# utils.py itself lands verbatim in Step 4); iid assumption.
H = float(-np.sum(probabilities * np.log2(probabilities)))
print(f"Shannon entropy (iid): {H:.4f} bits per sample")
print(f"Lower bound for the whole file:  n·H = {H * len(data):.0f} bits ≈ {H * len(data) / 8:.0f} bytes.")


## Step 4 — Build the (binary) Huffman codewords from scratch

The next two cells write the actual implementation files from the repository (`utils.py`, `huffman.py`).
The algorithm: give each distinct sample value its own node, then repeatedly merge the two
least-probable groups, appending a `0` bit to one and a `1` bit to the other (bits kept in reverse
order so a merge only touches the nodes being merged). When one group remains, the table is read out.
See the comments in `huffman.py` for the full story — no `huffman.compress()` here.


In [ ]:
%%writefile utils.py
import csv
from collections.abc import Sequence
from typing import Protocol, Iterable

from numpy import isclose, log2, ndarray, sum, uint8


# Any Symbol is one that has a string representation; this is a wrapper for type hints
class Symbol(Protocol):
    def __str__(self) -> str: ...


# [f]loating [p]oint [g]reater [t]han; ensures that a > b and not close to b
def fp_gt(a: float, b: float):
    return (a > b) and not isclose(a, b)


# The usual Shannon entropy
def entropy(pmf: ndarray) -> float:
    return sum(-pmf * log2(pmf))


def bytes2str(byte_vals: bytes):
    return "".join([str(i) for i in byte_vals])


def int2bytearray(n: int | uint8):
    return bytearray((n >> i) & 1 for i in range(int(n).bit_length() - 1, -1, -1))


def export_tuples_to_csv(
    data: Iterable[Sequence[object]], fname: str, header: Sequence[str]
):
    # Keep it to first row; doing all rows would require more iterations
    first_row = next(iter(data))
    if len(first_row) != len(header):
        raise ValueError(
            f"Length mismatch between provided headers and first row of data"
        )

    with open(fname, "w", newline="", encoding="utf-8") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(header)  # Write header first
        writer.writerow(first_row)  # Write first row
        writer.writerows(
            data
        )  # Write the rest of the data that does not include first row


In [ ]:
%%writefile huffman.py
from dataclasses import dataclass, field
from numpy import ndarray, isclose, uint8
from collections.abc import Sequence
from utils import fp_gt, int2bytearray
from functools import reduce

# @dataclass is just syntax sugar that avoids manual __init__ definitions
# @property is useful for getters and avoids the whole `class.prop()` business with trailing parens


@dataclass
class _Entry:
    _symbol: uint8
    _probability: float

    @property
    def symbol(self) -> uint8:
        return self._symbol

    @property
    def probability(self) -> float:
        return self._probability

    # bit-stack assigned to x; to have symbols pushed on top before reading it out upon giving the code
    # Put in __post_init__ to avoid having it as dataclass constructor argument
    def __post_init__(self):
        self._reverse_order_codeword_symbols: bytearray = bytearray()

    # Used when moving forward in Huffman; add a bit
    # NOTE: add_bit should be called something else if not binary
    def add_bit(self, addl_bit: int):
        assert addl_bit < 2, f"Attempted to add {addl_bit} >= 2 as a bit"
        self._reverse_order_codeword_symbols.append(addl_bit)

    # When done, go backwards and read bits to produce Huffman code
    @property
    def codeword(self) -> int:
        assert len(self._reverse_order_codeword_symbols) > 0
        # `a << 1 | b` shifts a left and puts b in; requires a=1 initially to avoid collapse
        return reduce(
            lambda a, b: (a << 1) | b,
            reversed(self._reverse_order_codeword_symbols + bytearray([1])),
        )
        # Will be addressed in `.get_final_code()` in HuffmanLayer


@dataclass
class _Aggregator:
    _entries: list[_Entry] = field(default_factory=list)
    # _probability: float = 0

    @property
    def entries(self):
        return self._entries

    @property
    def probability(self):
        return sum(entry.probability for entry in self._entries)

    # Summing two aggregators groups their entries together and sums probabilities
    def __add__(self, other: object):
        if isinstance(other, _Aggregator):
            return _Aggregator(self._entries + other._entries)
        raise NotImplementedError("Cannot add Aggregators to non-Aggregator objects")

    # For validation logic after class construction
    def __post_init__(self):
        # If greater than 1 *and* "far" from 1
        if fp_gt(self.probability, 1):
            raise RuntimeError(
                f"Aggregator probability of {self.probability:.2f} greater than 1"
            )

    # NOTE: add_bit should be called something else if not binary
    def add_bit(self, bit: int):
        for entry in self._entries:
            entry.add_bit(bit)


@dataclass
class _HuffmanLayer:
    # NOTE: Initialization of Huffman Layer would just put each Entry in its own
    # Aggregator; for D>2 case, may need dummy variables

    # The `field()` business makes mutable defaults ok
    _aggregators: list[_Aggregator] = field(default_factory=list)

    # We sort after initializing
    def __post_init__(self):
        self._sort()

    # reverse=True uses descending probability order; .pop() then removes the least probable aggregators
    def _sort(self):
        self._aggregators.sort(key=lambda x: x.probability, reverse=True)

    def step(self):
        if len(self._aggregators) == 1:
            raise StopIteration("Only one Aggregator in layer; no steps left")

        # Set to hold the combined Aggregators
        least_probable_aggregators = _Aggregator([])

        # NOTE: here, D=2, so we get bit = 0,1. In general would change the range int
        for bit in range(2):
            # Remove the least_probable
            least_probable_aggregator = self._aggregators.pop()

            # Add the bit
            least_probable_aggregator.add_bit(int(bit))

            # Put it into the container above
            least_probable_aggregators += least_probable_aggregator

        # Now feed the merged Aggregator back into the collection to get the new layer
        self._aggregators.append(least_probable_aggregators)

        # Sort it again before the next step
        self._sort()

    # Print the Huffman code as dict and remove 1 as MSB; raises error if HuffmanLayer is not singleton
    def get_final_code(self) -> dict[uint8, bytes]:
        if len(self._aggregators) > 1:
            raise RuntimeError(
                f"{len(self._aggregators)} > 1 aggregators left; need {len(self._aggregators) - 1} more `.step()`s"
            )

        # We sort by shortest lengths to longest lengths
        final_entries = sorted(
            self._aggregators[0].entries,
            key=lambda entry: entry.codeword,
        )

        def sentinel_to_bytes(n: int):
            # Remove the sentinel MSB after coercing into bytearray form
            return bytes(int2bytearray(n)[1:])

        final_code = {
            entry.symbol: sentinel_to_bytes(entry.codeword) for entry in final_entries
        }
        return final_code


def binary_huffman(symbols: Sequence[uint8], probabilities: ndarray):
    # Probabilities must be a PMF
    if not isclose(s := probabilities.sum(), 1):
        raise ValueError(f"Probabilities have sum {s} != 1")

    # Lengths must match
    if (len(symbols)) != (probabilities.size):
        raise ValueError(
            f"{len(symbols)} number of symbols neq to {probabilities.size} number of probabilities"
        )

    # Quickly return trivial codes
    if len(symbols) == 1:
        return {symbols[0]: b"0"}
    if len(symbols) == 2:
        return {symbols[0]: b"0", symbols[1]: b"1"}

    # Now the fun part!

    # Initialize the HuffmanLayer with singleton Aggregators
    huffman_instance = _HuffmanLayer(
        [_Aggregator([_Entry(x, p)]) for x, p in list(zip(symbols, probabilities))]
    )

    # Keep on taking Huffman `.step()`s until we hit the StopIteration
    while True:
        try:
            huffman_instance.step()
        except StopIteration:
            break

    return huffman_instance.get_final_code()


In [ ]:
from huffman import binary_huffman

# The table: sample value -> codeword (prefix-free by construction)
encoder = binary_huffman(list(symbols), probabilities)
decoder = {codeword: sym for sym, codeword in encoder.items()}

lengths = {len(c) for c in encoder.values()}
print(f"{len(encoder)} codewords | lengths range: {min(lengths)}–{max(lengths)} bits")
for sym in list(encoder)[:8]:
    print(f"  sample value {int(sym):3d} -> {''.join(map(str, encoder[sym]))}")
print("  ...")


## Step 5 — Encode the data; compare with the entropy bound

Replace each sample by its codeword and pack the bit stream into bytes.
The expected length per sample $\bar{L} = \sum p_i \ell_i$ must satisfy $H \leq \bar{L} < H + 1$.


In [ ]:
from math import ceil
from itertools import chain

expected_length = sum(len(encoder[sym]) * empirical[sym] for sym in encoder)
print(f"Expected length per sample: {expected_length:.4f} bits   (entropy bound: {H:.4f})")
print(f"Redundancy: {expected_length - H:.4f} bits/sample")

# Encode: codewords in sample order, then pack the bits into bytes
encoded_bits = bytearray(b"".join(chain(map(lambda datum: encoder[datum], data))))
compressed_bytes = ceil(len(encoded_bits) / 8)
print(f"\nEncoded bit stream: {len(encoded_bits)} bits -> {compressed_bytes} bytes")
print(f"Original file:     {len(data)} bytes")
print(f"Compression ratio: {compressed_bytes / len(data):.2f}")
print(f"Entropy bound for the whole file: {H * len(data) / 8:.0f} bytes "
      f"(we use {compressed_bytes}, i.e. {(compressed_bytes - H * len(data) / 8) / compressed_bytes * 100:.1f}% above)")


## Step 6 — Decode and verify

Greedy left-to-right decoding: accumulate bits until the running buffer matches a codeword
(prefix-freeness makes this unambiguous), emit the symbol, reset the buffer. Then compare with the original.


In [ ]:
import numpy as np

buffer = bytearray()
decoded = []
for bit in encoded_bits:
    buffer.append(bit)
    value = decoder.get(bytes(buffer), None)
    if value is None:
        continue          # not a complete codeword yet
    decoded.append(value)
    buffer = bytearray()

assert len(data) == len(decoded), "Length mismatch between original and decoded data"
assert np.all(data == decoded), "Decoded data differs from original"
print(f"Decoded {len(decoded)} samples.")
print("Every sample matches the original — the code is lossless. ✓")


## (Optional) Look at the data and the code

A picture of why Huffman works here: quiet passages make a few sample values dominate, so the
entropy is low and short codewords carry most of the probability mass.


In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.vlines(symbols, 0, probabilities, color="tab:blue")
ax1.set_yscale("log")
ax1.set_xlabel("sample value"); ax1.set_ylabel("probability")
ax1.set_title("Empirical distribution (log scale)")

code_lens = [len(encoder[sym]) for sym in symbols]
ax2.hist(code_lens, bins=range(min(code_lens), max(code_lens) + 2), weights=probabilities,
         align="left", color="tab:green", edgecolor="black")
ax2.set_xlabel("codeword length (bits)"); ax2.set_ylabel("probability mass")
ax2.set_title("Huffman codeword lengths")
plt.tight_layout(); plt.show()


---
**Where this lives:** the same code (plus `preprocess.py` and `__init__.py`, which run this exact
pipeline headlessly) is in the repository `elwais/646-huffman-implementation` (private) and in the
deliverable zip. The print-outs above are the "test run" evidence required by the assignment.
